##**Tarea 3. Proyecto de Detector Automatizado de Oportunidades de Inversión**

In [ ]:
import pandas as pd

###**1. Carga y Limpieza de Datos**

Cargar el archivo CSV:

In [ ]:
df = pd.read_csv("sample_data/ecom_rfm_data.csv")
print(df)

       InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER       6.0   
1         536365     71053                  WHITE METAL LANTERN       6.0   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER       8.0   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE       6.0   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.       6.0   
...          ...       ...                                  ...       ...   
106498    545333     21399             BLUE POLKADOT COFFEE MUG       2.0   
106499    545333     21454                 PAINTED PINK RABBIT        3.0   
106500    545333     21508              VINTAGE KID DOLLY CARD        1.0   
106501    545333     21539            RED RETROSPOT BUTTER DISH       1.0   
106502        54       NaN                                  NaN       NaN   

             InvoiceDate  UnitPrice  CustomerID         Country  
0        

Detectar y tratar valores nulos.

In [ ]:
print(df.isnull().sum())

InvoiceNo          0
StockCode          1
Description      305
Quantity           1
InvoiceDate        1
UnitPrice          1
CustomerID     36410
Country            1
dtype: int64


In [ ]:
df = df.dropna()

In [ ]:
print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


Convertir tipos de datos incorrectos.

In [ ]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
print(df.dtypes)

InvoiceNo       object
StockCode       object
Description     object
Quantity       float64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print(df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity              float64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object


Eliminar duplicados y filtrar valores atípicos.

In [ ]:
df = df.drop_duplicates()
print("Número de filas después de eliminar duplicados:", len(df))

Número de filas después de eliminar duplicados: 69128


In [ ]:
limite_cantidad = df["Quantity"].quantile(0.99)
limite_precio = df["UnitPrice"].quantile(0.99)

df = df[
    (df["Quantity"] <= limite_cantidad) &
    (df["UnitPrice"] <= limite_precio)
]
print("Filas después de eliminar valores extremos:", len(df))

Filas después de eliminar valores extremos: 67793


###**2. Análisis Exploratorio y Agrupaciones**

Agrupar los datos por categorías clave.

In [ ]:
ventas_pais = df.groupby("Country")["Quantity"].sum()
print(ventas_pais)

Country
Australia                 6963.0
Austria                    340.0
Belgium                   3619.0
Channel Islands            858.0
Cyprus                    2949.0
Czech Republic             346.0
Denmark                    668.0
EIRE                     10070.0
Finland                   3409.0
France                   18746.0
Germany                  18896.0
Greece                     526.0
Iceland                    634.0
Israel                     230.0
Italy                     1481.0
Japan                     5415.0
Lebanon                    386.0
Lithuania                  652.0
Netherlands              23004.0
Norway                    3901.0
Poland                     725.0
Portugal                  3532.0
Saudi Arabia                80.0
Singapore                 1089.0
Spain                     4524.0
Sweden                    1055.0
Switzerland               4798.0
United Arab Emirates       496.0
United Kingdom          439974.0
Name: Quantity, dtype: float64


Calcular métricas estadísticas avanzadas

In [ ]:
estadisticas = df.groupby("Country")["UnitPrice"].agg(
    ["mean", "median", "std", "count"]
)
print(estadisticas)

                          mean  median       std  count
Country                                                
Australia             3.263226   2.550  2.401729    217
Austria               2.064074   1.650  1.646204     27
Belgium               2.818211   1.690  3.037108    285
Channel Islands       5.273455   4.250  3.736111     55
Cyprus                3.439506   2.550  2.972070    243
Czech Republic        1.926000   1.490  1.267533     15
Denmark               3.181579   2.600  2.758080     38
EIRE                  3.423654   2.550  2.873285    843
Finland               2.793108   1.950  2.587573    222
France                2.592820   1.650  2.476192   1525
Germany               2.917367   1.690  2.889910   1667
Greece                6.480937   6.750  3.346151     32
Iceland               2.712667   2.525  1.999451     60
Israel                3.047778   1.950  2.857575     18
Italy                 2.841718   2.100  2.247678    163
Japan                 1.899605   1.450  1.448699

Crear nuevas columnas calculadas

In [ ]:
df["TotalVenta"] = df["Quantity"] * df["UnitPrice"]
print(df[["Description", "Quantity", "UnitPrice", "TotalVenta"]].head())

                           Description  Quantity  UnitPrice  TotalVenta
0   WHITE HANGING HEART T-LIGHT HOLDER       6.0       2.55       15.30
1                  WHITE METAL LANTERN       6.0       3.39       20.34
2       CREAM CUPID HEARTS COAT HANGER       8.0       2.75       22.00
3  KNITTED UNION FLAG HOT WATER BOTTLE       6.0       3.39       20.34
4       RED WOOLLY HOTTIE WHITE HEART.       6.0       3.39       20.34


In [ ]:
facturacion_pais = df.groupby("Country")["TotalVenta"].sum()
facturacion_pais = facturacion_pais.sort_values(ascending=False)
print(facturacion_pais)

Country
United Kingdom          901476.77
Netherlands              35707.71
Germany                  35205.89
France                   31768.01
EIRE                     24839.29
Australia                15824.78
Spain                     9485.05
Japan                     9401.68
Switzerland               7618.82
Portugal                  6411.18
Finland                   5720.54
Cyprus                    5083.02
Belgium                   4970.61
Norway                    4482.37
Italy                     2882.36
Greece                    2661.24
Channel Islands           2388.97
Singapore                 1803.07
Lebanon                   1693.88
Lithuania                 1661.06
Denmark                   1500.72
Poland                    1465.43
Sweden                    1428.36
Iceland                   1187.18
United Arab Emirates       889.24
Austria                    575.40
Israel                     551.66
Czech Republic             549.26
Saudi Arabia               145.92
Name: 

###**3. El "Motor de IA" basado en Reglas Estadísticas**

Definir un umbral inteligente usando la estadística del dataset.

In [ ]:
estadisticas_productos = df.groupby("StockCode")["UnitPrice"].agg(["mean", "std"])
print(estadisticas_productos.head())

           mean  std
StockCode           
10002      0.85  0.0
10080      0.85  NaN
10120      0.21  0.0
10123C     0.65  NaN
10124A     0.42  0.0


In [ ]:
estadisticas_productos["Umbral"] = (estadisticas_productos["mean"] - estadisticas_productos["std"])

print(estadisticas_productos.head())

           mean  std  Umbral
StockCode                   
10002      0.85  0.0    0.85
10080      0.85  NaN     NaN
10120      0.21  0.0    0.21
10123C     0.65  NaN     NaN
10124A     0.42  0.0    0.42


Utilizar filtrado booleano avanzado en Pandas y creación de columnas condicionales con
funciones con apply()) para etiquetar automáticamente las filas como "Oportunidad".

In [ ]:
df["PrecioMedio"] = df["StockCode"].map(estadisticas_productos["mean"])
df["Umbral"] = df["StockCode"].map(estadisticas_productos["Umbral"])

print(df[["StockCode", "UnitPrice", "PrecioMedio", "Umbral"]].head())



  StockCode  UnitPrice  PrecioMedio    Umbral
0    85123A       2.55     2.859163  2.691415
1     71053       3.39     3.631875  3.461508
2    84406B       2.75     3.356800  2.885379
3    84029G       3.39     3.658571  3.501245
4    84029E       3.39     3.649259  3.477062


In [ ]:
def clasificar_precio(fila):
    if fila["UnitPrice"] < fila["Umbral"]:
        return "Oportunidad"
    elif fila["UnitPrice"] > fila["PrecioMedio"]:
        return "Sobrevalorado"
    else:
        return "Precio de Mercado"

In [ ]:
df["Clasificacion"] = df.apply(clasificar_precio, axis=1)

print(df[["StockCode", "UnitPrice", "Clasificacion"]].head(10))

  StockCode  UnitPrice  Clasificacion
0    85123A       2.55    Oportunidad
1     71053       3.39    Oportunidad
2    84406B       2.75    Oportunidad
3    84029G       3.39    Oportunidad
4    84029E       3.39    Oportunidad
5     22752       7.65    Oportunidad
6     21730       4.25    Oportunidad
7     22633       1.85    Oportunidad
8     22632       1.85    Oportunidad
9     84879       1.69  Sobrevalorado


In [ ]:
oportunidades = df[df["Clasificacion"] == "Oportunidad"]

print(oportunidades[["StockCode", "Description", "UnitPrice", "Clasificacion"]].head(10))

   StockCode                          Description  UnitPrice Clasificacion
0     85123A   WHITE HANGING HEART T-LIGHT HOLDER       2.55   Oportunidad
1      71053                  WHITE METAL LANTERN       3.39   Oportunidad
2     84406B       CREAM CUPID HEARTS COAT HANGER       2.75   Oportunidad
3     84029G  KNITTED UNION FLAG HOT WATER BOTTLE       3.39   Oportunidad
4     84029E       RED WOOLLY HOTTIE WHITE HEART.       3.39   Oportunidad
5      22752         SET 7 BABUSHKA NESTING BOXES       7.65   Oportunidad
6      21730    GLASS STAR FROSTED T-LIGHT HOLDER       4.25   Oportunidad
7      22633               HAND WARMER UNION JACK       1.85   Oportunidad
8      22632            HAND WARMER RED POLKA DOT       1.85   Oportunidad
24     22914         BLUE COAT RACK PARIS FASHION       4.95   Oportunidad


###**4. Generación de Reportes Automatizados.**

Ordenar los resultados obtenidos de mayor a menor atractivo.

In [ ]:
mejores_oportunidades = oportunidades.sort_values(by="DiferenciaPrecio", ascending=False)
print(mejores_oportunidades)

      InvoiceNo StockCode           Description  Quantity         InvoiceDate  \
16433    537659     22765       NEWSPAPER STAND      23.0 2010-12-07 16:43:00   
26230    538508     21463  MIRRORED DISCO BALL       10.0 2010-12-12 13:32:00   
29817    538827     84356        POMPOM CURTAIN      24.0 2010-12-14 12:59:00   
45689    540277     84356        POMPOM CURTAIN      36.0 2011-01-06 12:18:00   
43554    540096     84356        POMPOM CURTAIN      12.0 2011-01-04 15:43:00   
...         ...       ...                   ...       ...                 ...   
84476    543397     16011       ANIMAL STICKERS      24.0 2011-02-08 10:32:00   
12038    537362     16011       ANIMAL STICKERS      41.0 2010-12-06 12:34:00   
31978    539036     16011       ANIMAL STICKERS      24.0 2010-12-15 14:35:00   
52520    540798     16011       ANIMAL STICKERS     120.0 2011-01-11 12:11:00   
84201    543371     16011       ANIMAL STICKERS      48.0 2011-02-07 14:53:00   

       UnitPrice  CustomerI

Filtrar únicamente las mejores oportunidades encontradas.

In [ ]:
mejores_oportunidades = mejores_oportunidades[mejores_oportunidades["DiferenciaPrecio"] > 0]

print(mejores_oportunidades)

      InvoiceNo StockCode           Description  Quantity         InvoiceDate  \
16433    537659     22765       NEWSPAPER STAND      23.0 2010-12-07 16:43:00   
26230    538508     21463  MIRRORED DISCO BALL       10.0 2010-12-12 13:32:00   
29817    538827     84356        POMPOM CURTAIN      24.0 2010-12-14 12:59:00   
45689    540277     84356        POMPOM CURTAIN      36.0 2011-01-06 12:18:00   
43554    540096     84356        POMPOM CURTAIN      12.0 2011-01-04 15:43:00   
...         ...       ...                   ...       ...                 ...   
84476    543397     16011       ANIMAL STICKERS      24.0 2011-02-08 10:32:00   
12038    537362     16011       ANIMAL STICKERS      41.0 2010-12-06 12:34:00   
31978    539036     16011       ANIMAL STICKERS      24.0 2010-12-15 14:35:00   
52520    540798     16011       ANIMAL STICKERS     120.0 2011-01-11 12:11:00   
84201    543371     16011       ANIMAL STICKERS      48.0 2011-02-07 14:53:00   

       UnitPrice  CustomerI

Exportar el informe final limpio a un nuevo archivo CSV.

In [ ]:
mejores_oportunidades.to_csv("informe_final.csv", index=False, sep=";", encoding="utf-8-sig")

print("Informe final exportado correctamente.")

Informe final exportado correctamente.


Descargar el archivo csv en local, para visualizarlo.

In [ ]:
from google.colab import files
files.download("informe_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>